# BMKMANU S3000 - exp100

## Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
import sys
sys.path.append(os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(''))), 'BMK_S1000/harpy'))
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _bmkmanu import bmkmanu

## Create sdata

In [4]:
dataset_id = 'exp100'

path = f'E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/BSTMAtrix'
bc_path = f'e:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/bc2bc/{dataset_id}.bc'
config_prefix = 'exp100_28Feb2025'
technology = 'S3000'
output_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy/sdata.zarr'

pipeline_name = 'BSTMAtrix'
species = 'mouse'
tissue_annotation_file = f'E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/tissue_annotation/tissue.geojson'


In [5]:
# sdata = sd.read_zarr(output_path)

version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


In [ ]:
sdata = bmkmanu(
    path=path, 
    dataset_id=dataset_id,
    filtered_counts=False,
    add_in_tissue_column=True,
    levels=None, 
    technology=technology,
    bc_path=bc_path,
    output=output_path
)
sdata

In [6]:
sdata = sd.read_zarr(output_path)

version mismatch: detected: RasterFormatV02, requested: FormatV04


In [7]:
levels = [*sdata.tables]
print(f"{pipeline_name} levels: {levels}")

BSTMAtrix levels: ['level_1', 'level_18', 'level_2', 'level_3', 'level_4', 'level_5', 'level_6', 'level_7', 'level_9']


## Specify plotting parameters

In [8]:
DPI = 300
MAX_DIM_FOR_PLOTS = 5000

HE_image = f'{dataset_id}_he'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale2
figsize: (16.626666666666665, 16.666666666666668)
plots_path: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy\plots


## Plot H&E

In [ ]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

## Plot BSTMAtrix levels and tissue annotation

In [ ]:
for level in levels:
    gc.collect()
    with matplotlib.rc_context({'legend.markerscale': 10}):
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"{pipeline_name} {level} - {pipeline_name} tissue annotation"
        
        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=0.1, contour_px=5, outline_alpha=1, color='in_tissue', scale=best_scale
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=False,
            legend_fontsize=rel_fontsize(fig, 0.015), 
            legend_fontweight='bold', 
            legend_loc='right margin',
            na_in_legend=False,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_annotation_{level}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)

## Calculate pixel size

In [9]:
# Determine pixel size in microns
he_shape = hp.im._image._get_spatial_element(sdata, f'{dataset_id}_he').shape

he_width = he_shape[2]
he_height = he_shape[1]

microns_per_pixel_x = 6800 / he_shape[2]
microns_per_pixel_y = 6800 / he_shape[1]
print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")

microns_per_pixel = (microns_per_pixel_x + microns_per_pixel_y) / 2
print(f"Pixel size: {microns_per_pixel:.3f} µm")

# NOTE: This is not a great way to have to summise what the pixel size is. This was recommended by BMKGENE and should be good enough, but there is definitely the possibility that the capture area is not exactly 6.8 mm.

Pixel size: 0.341 µm (X), 0.340 µm (Y)
Pixel size: 0.340 µm


## Create bounding box for total capture area

In [12]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f'{dataset_id}_level_1'].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_level_1'], get_all=True),
    overwrite = True,
)

2025-08-25 14:16:13,584 - harpy.utils._io - WARNING - layer with name 'exp100_capture_area' already exists. Overwriting...


In [13]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_level_1', color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.03)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.01793722..1.0044843].


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' do disable this    
         behaviour.                                                                                                
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy\plots\capture_area_bbox.png


In [14]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size. However, it should be close.

Area of capture area: 46.21 mm²


In [15]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size

Dimensions of capture area bbox: 0.68 cm x 0.68 cm


## Add tissue annotation

In [ ]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_level_1'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

In [ ]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = level, 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

In [ ]:
# Check spots in tissue
for level in levels:
    gc.collect()
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = "Tissue annotation spots"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=1, color='in_tissue_annotation', table_layer=level, scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.02)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'qupath_annotation_spots_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

In [16]:
# Total number of spots in table
n_spots = len(sdata.tables['level_1'].obs)
print(f"Number of spots in table: {n_spots}")

Number of spots in table: 3285798


In [17]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(sdata.tables['level_1'].obs[sdata.tables['level_1'].X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero}")

Number of spots in table with at least one UMI: 3221086


In [18]:
# Number of spots in tissue
n_spots_in_tissue = len(sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue}")

Number of spots in tissue: 1379257


In [19]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = sdata.tables['level_1'].obs['in_tissue_annotation'] == 1
umis_mask = np.array(sdata.tables['level_1'].X.sum(axis=1)).flatten() > 0
spots_mask = in_tissue_mask & umis_mask
n_spots_in_tissue_not_zero = spots_mask.sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero}")

Number of spots in tissue with at least one UMI: 1369888


In [20]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 19.60 mm²


## Capture effeciency metrics

In [21]:
# Total number of genes
print(f"Total number of genes: {(sdata.tables['level_1'].X.sum(axis=0) > 0).sum()}")

Total number of genes: 27686


In [22]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].X.sum(axis=0) > 0).sum()}")

Number of genes in tissue: 27432


In [23]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {sdata.tables['level_1'].X.sum()}")

Total number of unique UMIs: 448928032.0


In [24]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].X.sum()}")

Number of unique UMIs in tissue: 405777248.0


In [25]:
umi_gene = pd.read_csv(os.path.join(path, config_prefix, 'umi_gene.tsv.gz'), sep='\t', header=None)
umi_gene

,0,1,2,3
0,V2Bbc1_3-V2Abc2_379-V2Bbc3_191,CGAACCCTTC,ENSMUSG00000094793,2
1,V2Bbc1_3-V2Abc2_379-V2Bbc3_191,ATGCACCGGC,ENSMUSG00000076441,1
2,V2Bbc1_3-V2Abc2_379-V2Bbc3_191,TCCCTCTCGA,ENSMUSG00000042289,1
3,V2Bbc1_3-V2Abc2_379-V2Bbc3_191,GGGCAGCTAA,ENSMUSG00000022877,2
4,V2Bbc1_3-V2Abc2_379-V2Bbc3_191,TCTAGGACTC,ENSMUSG00000009927,2
...,...,...,...,...
448928263,V2Abc1_56-V2Abc2_354-V2Abc3_167,ACTGTTACGTTT,ENSMUSG00000029368,2
448928264,V2Abc1_56-V2Abc2_354-V2Abc3_167,CTTATATTGAAG,ENSMUSG00000078688,1
448928265,V2Abc1_56-V2Abc2_354-V2Abc3_167,GGGGCCAGTGCA,ENSMUSG00000078686,2
448928266,V2Abc1_56-V2Abc2_354-V2Abc3_167,GCCGCACGATTA,ENSMUSG00000029918,1


In [26]:
# Total number of UMIs
print("Total number of UMIs:", umi_gene[3].sum())

Total number of UMIs: 848833329


In [27]:
# Total number of unique barcodes
print("Total number of unique barcodes:", umi_gene[0].nunique())

Total number of unique barcodes: 3221086


In [6]:
barcode_pos = pd.read_csv(os.path.join(path, config_prefix, 'barcode_pos.tsv'), sep='\t', header=None)
barcode_pos

,0,1,2,3,4
0,V2Bbc1_332-V2Abc2_248-V2Abc3_281,0,0,0,12
1,V2Abc1_50-V2Abc2_209-V2Abc3_289,0,0,0,14
2,V2Bbc1_154-V2Abc2_121-V2Abc3_39,0,0,0,24
3,V2Abc1_344-V2Bbc2_29-V2Bbc3_356,0,0,0,26
4,V2Abc1_115-V2Abc2_183-V2Bbc3_157,0,0,0,27
...,...,...,...,...,...
3285793,V2Bbc1_188-V2Abc2_255-V2Abc3_45,42,41,50,38
3285794,V2Bbc1_89-V2Bbc2_340-V2Bbc3_142,42,41,50,39
3285795,V2Bbc1_335-V2Bbc2_337-V2Abc3_298,42,41,50,40
3285796,V2Abc1_334-V2Abc2_194-V2Abc3_355,42,41,50,41


In [29]:
features = pd.read_csv(os.path.join(path, config_prefix, 'features.tsv'), sep='\t', header=None)
features

,0,1,2
0,ENSMUSG00000051951,Xkr4,Gene Expression
1,ENSMUSG00000089699,Gm1992,Gene Expression
2,ENSMUSG00000102331,Gm19938,Gene Expression
3,ENSMUSG00000102343,Gm37381,Gene Expression
4,ENSMUSG00000025900,Rp1,Gene Expression
...,...,...,...
32280,ENSMUSG00000095523,AC124606.1,Gene Expression
32281,ENSMUSG00000095475,AC133095.2,Gene Expression
32282,ENSMUSG00000094855,AC133095.1,Gene Expression
32283,ENSMUSG00000095019,AC234645.1,Gene Expression


In [7]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in barcode_pos[0]:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].obs['barcode_name']:
        f.write(f"{item}\n")      

## Export unique UMI counts per gene

In [30]:
adata_in_tissue = sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1]

gene_counts = np.array(adata_in_tissue.X.sum(axis=0)).flatten()

df_gene_counts = pd.DataFrame({
    "gene": adata_in_tissue.var_names,
    "total_counts": gene_counts,
    "total_counts_corrected": gene_counts / overlap_area_mm2
})

df_gene_counts.to_csv(os.path.join(os.path.dirname(output_path), "gene_total_counts.csv"), index=False)

## QC

In [ ]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    gc.collect()
    sdata = qc_metrics(
        sdata,
        level,
        species
    )

In [40]:
# Plot QC metrics
for level in [level for level in levels if level not in ['level_1', 'level_2', 'level_3', 'level_4', 'level_18']]:
    gc.collect()
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()
        values = sdata.tables[level].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"scanpy - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=qc_metric, scale=best_scale, table_name=level, cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))
        
        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'scanpy_{qc_metric}_{level}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.01793722..1.0044843].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy\plots\scanpy_n_genes_by_counts_level_5.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.01793722..1.0044843].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy\plots\scanpy_total_counts_level_5.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.01793722..1.0044843].


...

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy\plots\scanpy_pct_counts_in_top_50_genes_level_9.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.01793722..1.0044843].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy\plots\scanpy_total_counts_mt_level_9.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.01793722..1.0044843].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp100/harpy\plots\scanpy_pct_counts_mt_level_9.png
